In [1]:
import sys, os

if os.path.abspath('MedViT') not in sys.path:
    sys.path.insert(0, os.path.abspath('MedViT'))

import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)
import gdown

from MedViT import MedViT_small, MedViT_base, MedViT_large

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

/home/vlad/miniconda3/envs/proiect/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder('dataset_split/train', transform=train_transforms)
val_dataset   = datasets.ImageFolder('dataset_split/val',   transform=test_transforms)
test_dataset  = datasets.ImageFolder('dataset_split/test',  transform=test_transforms)

loader_train = DataLoader(train_dataset, batch_size=12, shuffle=True,  num_workers=2, pin_memory=True)
loader_val   = DataLoader(val_dataset,   batch_size=12, shuffle=False, num_workers=2, pin_memory=True)
loader_test  = DataLoader(test_dataset,  batch_size=12, shuffle=False, num_workers=2, pin_memory=True)

classes     = train_dataset.classes
num_classes = len(classes)
print(f"Classes ({num_classes}): {classes}")
print(f"Train batches: {len(loader_train)}  Val: {len(loader_val)}  Test: {len(loader_test)}")

Classes (5): ['F0', 'F1', 'F2', 'F3', 'F4']
Train batches: 369  Val: 79  Test: 80


In [3]:
GDRIVE_IDS = {
    'small': '14wcH5cm8P63cMZAUHA1lhhJgMVOw_5VQ',
    'base':  '1Lrfzjf3CK7YOztKa8D6lTUZjYJIiT7_s',
    'large': '1sU-nLpYuCI65h7MjFJKG0yphNAlUFSKG',
}

CONSTRUCTORS = {
    'small': MedViT_small,
    'base':  MedViT_base,
    'large': MedViT_large,
}

def build_model(variant: str) -> nn.Module:
    """Download pretrained ImageNet weights, replace head, return model on device."""
    ckpt_path = f'medvit_{variant}_imagenet.pth'
    if not os.path.exists(ckpt_path):
        print(f"Downloading MedViT-{variant} checkpoint...")
        gdown.download(id=GDRIVE_IDS[variant], output=ckpt_path, quiet=False)

    model = CONSTRUCTORS[variant](num_classes=num_classes)

    state = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    if isinstance(state, dict) and 'model' in state:
        state = state['model']

    head_keys = [k for k in state if 'proj_head' in k]
    for k in head_keys:
        del state[k]

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"MedViT-{variant} loaded. Head re-initialised from scratch: {head_keys}")

    return model.to(device)

In [4]:
criterion = nn.CrossEntropyLoss()
os.makedirs('checkpoints', exist_ok=True)

def train_model(model: nn.Module, optimizer, num_epochs: int = 20, name: str = 'model') -> nn.Module:
    since = time.time()
    for epoch in range(num_epochs):
        print(f"Epoch {epoch + 1}/{num_epochs}")
        print("-" * 10)
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                loader = loader_train
            else:
                model.eval()
                loader = loader_val

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss     += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(loader.dataset)
            epoch_acc  = running_corrects.double() / len(loader.dataset)
            print(f"{phase} Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f}")

        if (epoch + 1) % 5 == 0:
            ckpt = f'checkpoints/{name}_epoch{epoch + 1}.pth'
            torch.save(model.state_dict(), ckpt)
            print(f"Checkpoint saved: {ckpt}")

    elapsed = time.time() - since
    print(f"Training complete in {elapsed // 60:.0f}m {elapsed % 60:.0f}s")
    return model

In [11]:
model_large = build_model('large')
optimizer_large = optim.Adam(model_large.parameters(), lr=0.0001)

print("=== Training MedViT-large ===")
model_large = train_model(model_large, optimizer_large, num_epochs=20, name='medvit_large')
torch.save(model_large.state_dict(), 'medvit_large.pth')
print("Saved medvit_large.pth")

initialize_weights...
MedViT-large loaded. Head re-initialised from scratch: ['proj_head.0.weight', 'proj_head.0.bias']
=== Training MedViT-large ===
Epoch 1/20
----------


OutOfMemoryError: CUDA out of memory. Tried to allocate 14.00 MiB. GPU 0 has a total capacity of 7.63 GiB of which 5.00 MiB is free. Including non-PyTorch memory, this process has 7.60 GiB memory in use. Of the allocated memory 7.06 GiB is allocated by PyTorch, and 416.68 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
def evaluate_model(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)
            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    acc  = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred)
    return {
        'y_true': y_true, 'y_pred': y_pred,
        'accuracy': acc, 'precision_macro': prec,
        'recall_macro': rec, 'f1_macro': f1,
        'confusion_matrix': cm,
    }

In [ ]:
results = {}

variant = 'large'
model = build_model(variant)
model.load_state_dict(torch.load(f'medvit_{variant}.pth'))
model = model.to(device)

results[variant] = evaluate_model(model, loader_test)
print(f"\n=== MedViT-{variant} — Classification Report (test) ===")
print(classification_report(
    results[variant]['y_true'],
    results[variant]['y_pred'],
    target_names=classes,
    zero_division=0,
))

del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

sns.heatmap(
    results['large']['confusion_matrix'],
    annot=True, fmt='d', cmap='Greens',
    xticklabels=classes, yticklabels=classes,
    ax=ax,
)
ax.set_title('MedViT-large')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')

plt.suptitle('Confusion Matrix — MedViT-large', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
metrics = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']

print("\n=== Summary ===")
print(f"{'Variant':<12} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 52)
variant = 'large'
r = results[variant]
print(f"{variant:<12} {r['accuracy']:>10.4f} {r['precision_macro']:>10.4f} {r['recall_macro']:>10.4f} {r['f1_macro']:>10.4f}")